# Five modelling decisions

This page is the second step of the beginner route, for readers who have run the
Quickstart and want to know which choices their model makes. It explains the five choices
that specify a boosting model: the quantity to predict, a family compatible with
the response, candidate effect shapes, a stopping iteration, and diagnostic
checks. The [Quickstart](quickstart.ipynb) made each choice with a default; this
page states what each default assumes and when to change it. It assumes the
Quickstart has been run. The next pages, [glmboost](glmboost.ipynb) and
[gamboost](gamboost.ipynb), document the two fitting functions in full.

## 1. Target quantity

Choose the quantity the analysis needs to report, such as a mean, an event
probability, or a quantile. The same response can support several such targets.

| If you need | The target is | Where to go |
|---|---|---|
| a typical value | the conditional **mean** | `Gaussian`, the default |
| a probability, and you will use the number | a **probability** | `Binomial`; check it is calibrated |
| a class label only | a **decision rule**, and calibration does not matter | `Binomial` or `AdaExp` |
| a rate or a count | a **mean count** | `Poisson`, or `NBinomial` when the spread exceeds the mean |
| a value that some fraction falls below | a **quantile** | `Quantile(tau=...)` |
| time until an event, with some not yet observed | a **survival** distribution | `CoxPH`, `Weibull`, `Loglog`, `Lognormal` |
| an ordering, not a level | a **rank** | `Gehan`, `Cindex`, `AUC` |
| the spread or shape, not only the level | a **distribution parameter** | `glmboostLSS` / `gamboostLSS` |

A mean model alone does not estimate response spread or predictive centiles.
For a prediction interval, fit a distribution or use a calibrated interval
procedure. For a triage list, evaluate ranking; for a centile, use a quantile
model or a fitted distribution's quantile function.

## 2. Response type and family

The response's support and its censoring restrict the family before any
preference does. A family evaluated outside its domain returns a number that
means nothing, so the fit refuses instead:

```
family 'Binomial' rejected response: y must be in {-1, +1}, {0, 1},
or [0, 1] proportions, got -1.33
```

- **Real line** — `Gaussian`, `Laplace`, `Huber`, `Quantile`, `Expectile`.
- **{0, 1} or proportions** — `Binomial`, `AdaExp`, `HingeLoss`.
- **Non-negative integers** — `Poisson`, `NBinomial`.
- **Positive reals** — `GammaReg`, and the AFT survival families.
- **Ordered levels 1 to K** — `PropOdds`.
- **Unordered classes** — `Multinomial` (`gamboost` only).
- **Time with censoring** — the survival families, via `Surv(time, status)`.
  Dropping the censoring and fitting a classifier answers a different
  question, which [](../cases/wpbc_censoring.ipynb) works through.

## 3. Term shape and base learner

A base learner is a hypothesis about the shape of one effect. The penalty
decides what it shrinks *toward*, and that "toward" is the term's null space —
the part no penalty removes.

| Learner | Shape | Shrinks toward | Wrong when |
|---|---|---|---|
| `bols` | a straight line | zero slope | the effect bends |
| `bbs` | a smooth curve | a straight line | the effect has a genuine jump |
| `bns` | a smooth curve | zero everywhere | you wanted the straight-line anchor |
| `btree` | a step | zero | the effect is gradual |
| `brandom` | a mean per level | the overall mean | levels are few and well observed |
| `bmono` | a curve that cannot reverse | a monotone fit | the direction is not known from mechanism |

One learner is selected per iteration by its fit to the current negative
gradient. A more flexible candidate can win because its larger function space
fits more patterns. Give every candidate the same `df` when comparing their
predictive contributions;
[](../cases/learner_flexibility.ipynb) measures what happens otherwise — a
term with no effect at all gets selected.

## 4. Choosing `mstop`

`mstop` limits the number of updates and therefore controls the fitted
model's complexity together with `nu` and the learners' penalties. Choose it
using held-out risk:

```python
cvr = fit.cvrisk()          # held-out risk along the path
final = fit[:cvr.optimal_mstop]
```

The minimum of this curve is optimistic as an estimate of the selected model's
performance because the folds also chose `mstop`. `cvr.mstop_1se` offers an
earlier stop whose estimated risk is within one standard error of the minimum;
evaluate either selected model on separate test data before reporting its
predictive performance. If risk is still falling at the last iteration, extend
the path: the reported optimum is limited by the fitted budget. `cvrisk` warns
when this happens.

**The fold has to match the data.** Rows that are not exchangeable need a
scheme that says so, or the held-out risk is optimistic:

```python
mb.grouped_folds(subject_id, B=5)   # repeated measurements
mb.stratified_folds(y, B=10)        # a rare class
mb.rolling_folds(n, B=5)            # a time series
```

`scheme.check_leakage(groups=...)` will tell you whether a fold matrix you
already have splits a group across the boundary.

## 5. Diagnostics for each assumption

Use the following diagnostics to assess the chosen distribution, effect shapes,
validation design and intervals. Each check addresses a different part of the
analysis; selection frequency, for example, measures stability under resampling
rather than establishing that a variable has a true effect.

| Assumption | What can falsify it |
|---|---|
| the distribution | `fit.pit()`, `fit.worm_data()` — a PIT histogram that is not flat |
| the fitted shape | `fit.partial_effects()` against a more flexible refit |
| independence | `mb.acf_residuals(fit)` — structure left in the residual ordering |
| the interval's coverage | `mb.coverage(...)` on held-out rows |
| that the input is in range | `fit.range_report(newdata)` — see [](../cases/beyond_the_data.ipynb) |
| that a selected term is real | `fit.stabsel(...)` — how often it survives resampling |

`fit.parity_notes()` lists where this fit differs from what its names suggest,
and says whether upstream `mboost` shares the difference.

A model restored from disk has the fitted path but lacks training data.
Consequently `cvrisk`, `stabsel`, `confint` and `conformal_interval` cannot
refit it. `fit.unavailable_operations()` lists unavailable operations and
the input missing for each.

## Reference chapters

The reference pages give arguments, examples and limitations for these choices:

- [](families.ipynb) — every loss, its link, and where it refuses.
- [](base_learners.ipynb) — every learner, its penalty and null space.
- [](model_selection.ipynb) — `cvrisk` in depth.
- [](diagnostics.ipynb) — the checks above, worked.
- [](interface_map.ipynb) — how the pieces fit together.
